# Imported required libraries

In [1]:
import pandas as pd
import numpy as np
import random
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gamma

In [2]:
# !pip install jupyter-ai   

# Generate dataset for analysis

In [3]:
# Set seed for reproducibility
np.random.seed(42)

n_samples = 400000

# 1. Age and Employment Length
# Balanced distribution between 21 and 65
# random.beta(2, 5) creates a distribution weighted toward the lower end
age_raw = np.random.beta(2, 5, size=n_samples)
person_age = (21 + (age_raw * (65 - 21))).astype(int)
# person_age = np.random.randint(21, 66, size=n_samples)
# emp_length is capped by (age - 21), maxed at 20
emp_length = np.array([min(np.random.randint(0, 30), a - 21) for a in person_age])

# 2. Income (Realistic UK Log-normal distribution)
# Median UK income is approx £33k-35k; we'll aim for a mean around £38k with a tail
annual_inc = np.random.lognormal(mean=10.5, sigma=0.4, size=n_samples).round(-2)

# 3. Logic-Based Purpose and Loan Amount
purposes = ['Debt Consolidation', 'Home Improvement', 'Car Purchase', 'Major Purchase', 'Small Business']
purpose_probs = [0.35, 0.22, 0.18, 0.10, 0.15]
purpose = np.random.choice(purposes, size=n_samples, p=purpose_probs)
loan_amount = np.zeros(n_samples)

# Assign Loan Amounts using Gamma Distribution
# Mean = Shape x Scale
for p in purposes:
    mask = (purpose == p)
    count = np.sum(mask)
    
    if p == 'Home Improvement':
        # Mean ~£60,000, 
        loan_amount[mask] = np.random.gamma(shape=15.0, scale=4000, size=count)
    elif p == 'Car Purchase':
        # Mean ~£30k
        loan_amount[mask] = np.random.gamma(shape=6.0, scale=5000, size=count)
    elif p == 'Small Business':
        # Mean ~£45k, High variance 
        loan_amount[mask] = np.random.gamma(shape=3.0, scale=15000, size=count)
    else:
        # Debt Consolidation / Major Purchase: Mean ~£16k
        loan_amount[mask] = np.random.gamma(shape=2.0, scale=8000, size=count)
        
# Apply global clipping and rounding to maintain bounds
loan_amount = np.clip(loan_amount, 5000, 150000).round(-2)

# 4. Loan-to-Value (LTV) and Risk Correlation
# Higher LTV leads to higher interest rates and lower grades, max ratio is 1
ltv = np.random.uniform(0.3, 1.0, size=n_samples)

# 5. Interest Rate & Bins
# Base rate + LTV factor + random noise
int_rate = (5.0 + (ltv * 8) + np.random.normal(0, 1, n_samples)).clip(6.4, 14.9)

bins = [0, 7.5, 9.5, 11.5, 15.0]
labels = ['6.4-7.5', '7.5-9.5', '9.5-11.5', '11.5-14.9']
interest_band = pd.cut(int_rate, bins=bins, labels=labels)

# 6. Loan Grade (A=Lowest risk, D=Highest risk)
# Correlation: Higher interest rate -> Worse grade
grade_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
grade_indices = np.digitize(int_rate, [8.0, 10.5, 12.5])
grade = [grade_map[i] for i in grade_indices]

# 7. Fixed Obligations and Existing Loans
# Fixed obligations usually scale with income (rent/mortgage)
fixed_obligation = (annual_inc * np.random.uniform(0.2, 0.45, n_samples) / 12).round(2)
existing_loans_exists = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.6, 0.4])
# Equated Monthly Instalment for principal and interest
existing_emi = np.where(existing_loans_exists == 'Yes', 
                        (loan_amount * np.random.uniform(0.01, 0.03, n_samples)), 0).round(2)

# 8. Discrete Categorical Variables
# 8.1 Cities
cities = ['London', 'Manchester', 'Birmingham', 'Leeds', 'Glasgow', 'Liverpool', 'Bristol', 'Edinburgh']
n_other_cities = len(cities) - 1

## Generate random weights for the "other" cities that sum to 0.68
## np.random.dirichlet([1]*n) creates n random numbers summing to 1
other_weights = np.random.dirichlet([1] * n_other_cities) * 0.68

## Combine London (0.32) with the random other weights
city_probs = np.insert(other_weights, 0, 0.32)
addr_cities = np.random.choice(cities, size=n_samples, p=city_probs)

# 8.2 Others
home_ownership = np.random.choice(['RENT', 'OWN', 'MORTGAGE', 'OTHER'], size=n_samples, p=[0.3, 0.1, 0.58, 0.02])
emp_title = np.random.choice(['Manager', 'Nurse', 'Engineer', 'Sales', 'Teacher', 'Self-Employed', 'Analyst', 'Techican', 'Accountant', 'Others'], size=n_samples)
term = np.random.choice([12, 24, 36, 48, 60, 72, 84], size=n_samples)
regularity_and_stability_of_inflows = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.90, 0.10])
collateral = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.6, 0.4])


## Assuming a person can only start a credit line at age 18.
max_cr_months = (person_age - 18) * 12
## Ensure we don't get negative values if someone is 18 (clip at 0)
max_cr_months = np.clip(max_cr_months, 0, None)

# Generate random months and cap them at the max possible for that specific age
mths_since_earliest_cr_line = np.random.randint(12, 400, size=n_samples)
mths_since_earliest_cr_line = np.minimum(mths_since_earliest_cr_line, max_cr_months)

# Generate two random weights that sum to 1 (e.g., [0.65, 0.35])
marriage_probs = np.random.dirichlet([55, 45]) # Higher numbers like 5,5 keep it relatively balanced but not 50/50
marriage_status = np.random.choice(['Yes', 'No'], size=n_samples, p=marriage_probs)

# 9. Gender Distribution
gender = np.random.choice(['Male', 'Female', 'Non-binary'], size=n_samples, p=[0.50, 0.45, 0.05])

# 10. Endogenous Default Status
# Logic: Collateral 'Yes' reduces default probability; No regular inflow increases it
collateral_effect = np.where(collateral == 'Yes', -0.2, 0.1) 
inflow_effect = np.where(regularity_and_stability_of_inflows == 'No', 0.3, 0)

# Calculate default probability
default_prob = (ltv * 0.4) + collateral_effect + inflow_effect

# Clips any values below 0 to 0 and any above 1 to 1
default_prob = np.clip(default_prob, 0, 1)

# Thresholding for binary result (Yes/No), and add a small amount of random 'noise' to simulate real-world unpredictability
default_status = np.where(default_prob + np.random.normal(0, 0.1, n_samples) > 0.5, 'Yes', 'No')

# Terms Remaining at Default
## Using Beta distribution (1.5, 4) to ensure default happens early-to-mid loan life
default_timing_pct = np.random.beta(1.5, 4, size=n_samples)
# terms_remaining = Total Term - (Total Term * % point of default)
terms_remaining = np.where(default_status == 'Yes', (term * (1 - default_timing_pct)).astype(int), 0)
# Ensure at least 1 term was remaining (can't default after paying it off)
terms_remaining = np.where((default_status == 'Yes') & (terms_remaining == 0), 1, terms_remaining)

# Create DataFrame
df = pd.DataFrame({
    'person_age': person_age,
    'gender': gender,
    'annual_inc': annual_inc,
    'emp_length': emp_length,
    'emp_title': emp_title,
    'home_ownership': home_ownership,
    'marriage_status': marriage_status,
    'addr_cities': addr_cities,
    'purpose': purpose,
    'loan_amount': loan_amount,
    'term': term,
    'terms_remaining': terms_remaining,
    'int_rate': int_rate,
    'interest_band': interest_band,
    'grade': grade,
    'loan_to_value': ltv,
    'collateral': collateral,
    'mths_since_earliest_cr_line': mths_since_earliest_cr_line,
    'regularity_of_inflows': regularity_and_stability_of_inflows,
    'fixed_obligation': fixed_obligation,
    'existing_loans': existing_loans_exists,
    'existing_emi': existing_emi,
    'default_status': default_status
})

# Display summary stats
print(f"\n% of loans <= £100k: {(df['loan_amount'] <= 100000).mean() * 100:.2f}%")
print(f"Default Rate: {(df['default_status'] == 'Yes').mean() * 100:.2f}%")
print(f"Avg Terms Remaining at Default (Defaults only): {df[df['default_status'] == 'Yes']['terms_remaining'].mean():.1f} months")
# print(df[['term', 'terms_remaining', 'default_status']].head(10))


% of loans <= £100k: 99.16%
Default Rate: 9.52%
Avg Terms Remaining at Default (Defaults only): 34.4 months


In [4]:
# Logic check:
logical_errors = df[df['mths_since_earliest_cr_line'] > (df['person_age'] * 12)].shape[0]
print(f"Logical Errors (Credit age > Actual age): {logical_errors}")

Logical Errors (Credit age > Actual age): 0


In [5]:
df.head()

,person_age,gender,annual_inc,emp_length,emp_title,home_ownership,marriage_status,addr_cities,purpose,loan_amount,...,interest_band,grade,loan_to_value,collateral,mths_since_earliest_cr_line,regularity_of_inflows,fixed_obligation,existing_loans,existing_emi,default_status
0,36,Female,70500.0,14,Analyst,RENT,Yes,Manchester,Debt Consolidation,27200.0,...,7.5-9.5,B,0.613280,Yes,27,Yes,1513.83,No,0.00,No
1,31,Male,26700.0,1,Accountant,MORTGAGE,Yes,Bristol,Home Improvement,51000.0,...,9.5-11.5,B,0.724140,Yes,156,Yes,981.82,No,0.00,No
2,39,Male,73500.0,18,Analyst,MORTGAGE,No,Manchester,Car Purchase,27600.0,...,6.4-7.5,A,0.383181,No,252,Yes,1332.15,Yes,487.02,No
3,28,Male,29000.0,7,Techican,RENT,Yes,Glasgow,Home Improvement,65500.0,...,7.5-9.5,B,0.362882,No,120,Yes,778.53,Yes,1135.30,No
4,45,Male,42900.0,8,Nurse,RENT,No,London,Debt Consolidation,16400.0,...,9.5-11.5,B,0.562121,No,81,Yes,1274.00,Yes,468.29,No


# Exporting the dataset

In [6]:
# Define designated file path
file_path = r"C:/Users/danie/Downloads/Data Analytic/credit_loan_analysis/credit_loan_dataset.csv"

# Save to CSV
try:
    # index=False prevents pandas from adding an extra column for the row numbers
    df.to_csv(file_path, index=False)
    
    # Verification
    if os.path.exists(file_path):
        print(f"Successfully saved {len(df):,} rows to: {os.path.abspath(file_path)}")
        print(f"File size: {os.path.getsize(file_path) / (1024*1024):.2f} MB")
    else:
        print("File was not saved. Check folder permissions.")

except Exception as e:
    print(f"An error occurred while saving: {e}")

Successfully saved 400,000 rows to: C:\Users\danie\Downloads\Data Analytic\credit_loan_analysis\credit_loan_dataset.csv
File size: 59.99 MB


# Exporting the notebook

In [7]:
import shutil

# Define your source and destination
current_notebook_name = "credit_loan (generated).ipynb" 
destination_path = r"C:\Users\danie\Downloads\Data Analytic\credit_loan_analysis"

# Ensure destination exists
if not os.path.exists(destination_path):
    os.makedirs(destination_path)

# Perform the copy
try:
    # This looks for the file in the current working directory
    shutil.copy(current_notebook_name, os.path.join(destination_path, current_notebook_name))
    print(f"✅ Notebook successfully copied to: {destination_path}")
except FileNotFoundError:
    print(f"❌ Error: Could not find '{current_notebook_name}'.")
    print("👉 Make sure the filename matches exactly and you have saved (Ctrl+S) recently.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Notebook successfully copied to: C:\Users\danie\Downloads\Data Analytic\credit_loan_analysis
